# 基礎編7
このノートブックでは、キーバリューコントラクトの基礎的な使い方を説明します。

## キーバリューコントラクトの特徴

- キーバリューコントラクトは、ブロックチェーン上のキーバリューストアです。
- インタフェースは通常のスマートコントラクトと共通化されています。
- さまざまな値を格納できます。 
- 他のコントラクトから呼び出して使用することもできます。  

## 準備

In [1]:
var { domain } = await import('../lib/load-config.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

## キーバリューコントラクトの基本的な操作

キーバリューコントラクトを作成します。

In [2]:
var resp = await rpc.call(adminWallet, 'c1create', { type: 'keyvalue', domain });
console.log(resp);
var kvid = resp.value;

{
  txno: 701715,
  txid: 'xexJubQ3NKZr5t4H2FqRpNQh2VRyrioFtx8iFLzcqTXGKB',
  status: 'ok',
  value: 'c210827874'
}


キーkey1にバリュー123をセットします。

In [3]:
var resp = await rpc.call(adminWallet, kvid, { cmd: 'set', key: 'key1', value: 123 });
console.log(resp);

{
  txno: 701716,
  txid: 'xNnX2i4RKMgTBy8yFrMVsFgJaCbQXedJ7NgubWq4nu2zMB',
  status: 'ok',
  value: null
}


キーkey1のバリューを取り出します。

In [4]:
var resp = await rpc.call(adminWallet, kvid, { cmd: 'get', key: 'key1' });
console.log(resp);

{
  txno: 701717,
  txid: 'xUJJjs5A7gapTT2a8jQTfZKem3VDLz6QctNeVgZCHnAiy',
  status: 'ok',
  value: 123
}


キーkey1のバリューに45を加算します。

In [5]:
var resp = await rpc.call(adminWallet, kvid, { cmd: 'add', key: 'key1', value: 45 });
console.log(resp);

{
  txno: 701718,
  txid: 'xzKm6Uoqpa9KF3kpYDrcN6Wp6Htd3i5AKqms3UWhMcsP4',
  status: 'ok',
  value: null
}


キーkey1のバリューを取り出します。  
加算の結果として、123+45=168を得ます。

In [6]:
var resp = await rpc.call(adminWallet, kvid, { cmd: 'get', key: 'key1' });
console.log(resp);

{
  txno: 701719,
  txid: 'xjGsm9GuumA6drHjv2CX4dYXgQgSqVQXUKLzgYn3Pe3Br',
  status: 'ok',
  value: 168
}


キーkey1を削除します。

In [7]:
var resp = await rpc.call(adminWallet, kvid, { cmd: 'delete', key: 'key1' });
console.log(resp);

{
  txno: 701720,
  txid: 'xhVkNrwW2WM3Fa6DwPyQ5aX3fQUvwaoLcMpPmwh2dQPFPB',
  status: 'ok',
  value: null
}


キーkey1のバリューを取り出します。  
キーは削除されているためnullが返ります。

In [8]:
var resp = await rpc.call(adminWallet, kvid, { cmd: 'get', key: 'key1' });
console.log(resp);

{
  txno: 701721,
  txid: 'xrYHodGuP7tHn7cCC8PsV3bEcrcqnD6i3QjeaNGCVUfZq',
  status: 'ok',
  value: null
}


## さまざまな値を格納する

キーバリューコントラクトにさまざまな値を格納します。  
JSONで表すことができる値を格納することができます。

In [9]:
await rpc.call(adminWallet, kvid, { cmd: 'set', key: 'key-0', value: 555 }); // number
await rpc.call(adminWallet, kvid, { cmd: 'set', key: 'key-1', value: "this is a string" }); // string
await rpc.call(adminWallet, kvid, { cmd: 'set', key: 'key-2', value: [1, 3, 5, 7] }); // array
await rpc.call(adminWallet, kvid, { cmd: 'set', key: 'key-3', value: { abc: 123, xyz: 'XXX' }}); // object

{
  txno: 701725,
  txid: 'xJdY23dZrnP7hUP2FCw9fhWxr88ipp3Eqagm9LBHUVRr8',
  status: 'ok',
  value: null
}

格納した値を取り出します。

In [10]:
for(var i=0; i<=3; i++){
    var resp = await rpc.call(adminWallet, kvid, { cmd: 'get', key: 'key-'+i });
    console.log('key-'+i, resp.value);
}

key-0 555
key-1 this is a string
key-2 [ 1, 3, 5, 7 ]
key-3 { abc: 123, xyz: 'XXX' }


## コントラクトの中から呼び出して使用する

キーバリューを参照する簡単なスマートコントラクトを作成します。  
下記のコントラクトは、kvidで指定するキーバリューコントラクトを開き、keyに対応するバリューを取り出して返します。

In [11]:
var cid_1 = await deploySmartContract({ kvid: 'string', key: 'string' }, function basic7_1(kvid, key){
    var kv = openContract(kvid);
    var value = kv.get(key);
    return value;
});

作成したコントラクトがキーバリューコントラクトにアクセスできるように権限を設定します。

In [12]:
await rpc.call(adminWallet, 'c1update', { id: kvid, prop: 'add accessible_to', value: [cid_1] });

{
  txno: 701733,
  txid: 'xEuizXTwW6vAi26yUDgCUoe7MnKsoUJkdGjcFRK6oGtAt',
  status: 'ok',
  value: null
}

作成したスマートコントラクトを呼び出します。  
key-1の値が返ります。

In [13]:
var resp = await rpc.call(adminWallet, cid_1, { kvid, key: 'key-1' });
console.log(resp);

{
  txno: 701734,
  txid: 'xJCBZHP95K7VgCDZzpSB2GpWDzRAZgFRqSLYJmt55ri38',
  status: 'ok',
  value: 'this is a string'
}


キーを変えてスマートコントラクトを呼び出します。  
key-2の値が返ります。

In [14]:
var resp = await rpc.call(adminWallet, cid_1, { kvid, key: 'key-2' });
console.log(resp);

{
  txno: 701736,
  txid: 'xSoSifGvcNcAajfhkfc7zpfLcYTSYsbLYymCUpL5VagFo',
  status: 'ok',
  value: [ 1, 3, 5, 7 ]
}


もうすこし複雑な例を示します。  
下記のコントラクトは、key-1のバリューにkey-0の値を加え、key-0を削除するという処理をします。（深い意味はありません）    
スマートコントラクトにまとめることにより、これらの一連の処理はアトミックに実行されます。

In [15]:
var cid_2 = await deploySmartContract({ kvid: 'string' }, function basic7_2(kvid){
    var kv = openContract(kvid);
    var value = kv.get('key-0');
    kv.delete('key-0');
    kv.add('key-1', value);
    return kv.get('key-1');
});

作成したコントラクトがキーバリューコントラクトにアクセスできるように権限を設定します。

In [16]:
await rpc.call(adminWallet, 'c1update', { id: kvid, prop: 'add accessible_to', value: [cid_2] });

{
  txno: 701741,
  txid: 'xTa3j2oeaf2xp7MD5HRCMxbeKksks5yGBnsUgYuHQT44AB',
  status: 'ok',
  value: null
}

作成したスマートコントラクトを呼び出します。  

In [17]:
var resp = await rpc.call(adminWallet, cid_2, { kvid });
console.log(resp);

{
  txno: 701742,
  txid: 'xyDTqew5mBaaMLzn6VNikado5ZGMn5jfGx3kALXWcr2t4',
  status: 'ok',
  value: 'this is a string555'
}


確認のため、key-0とkey-1のバリューを参照します。

In [18]:
for(var i=0; i<=1; i++){
    var resp = await rpc.call(adminWallet, kvid, { cmd: 'get', key: 'key-'+i });
    console.log('key-'+i, resp.value);
}

key-0 null
key-1 this is a string555


## キーバリューコントラクトのクリア

キーバリューコントラクト内に格納されたデータをすべて削除するには、deleteAllを使います。

In [19]:
var resp = await rpc.call(adminWallet, kvid, { cmd: 'deleteAll' });
console.log(resp);

{
  txno: 701749,
  txid: 'xsW6NWC4KLEQ2jVWymAcARLyRaZbF2dRhx3VooYho6nTq',
  status: 'ok',
  value: null
}


確認します。

In [20]:
for(var i=0; i<=3; i++){
    var resp = await rpc.call(adminWallet, kvid, { cmd: 'get', key: 'key-'+i });
    console.log('key-'+i, resp.value);
}

key-0 null
key-1 null
key-2 null
key-3 null


## クリーンアップ
このノートブックの中で作成したキーバリューコントラクトは今後使用しないので、削除しておきます。

In [21]:
var resp = await rpc.call(adminWallet, 'c1terminate', { id: kvid });
console.log(resp);

{
  txno: 701754,
  txid: 'xUG22gx65AfzvxzpjizKedwN2jyM34QB949eGLYUuxtmMB',
  status: 'ok',
  value: 'c210827874'
}
